# 🤖 Enterprise Chatbot Evaluation with LangSmith — Datasets, LLM-as-a-Judge & Automated Benchmarks

**Production Masterclass by Krish Naik | KRISHAI Technologies & Production Engineering**

Evaluating conversational chatbots is fundamentally different from traditional software testing. Because Large Language Models generate non-deterministic, open-ended responses, simple keyword checks and string comparisons fail. In enterprise production systems, **automated LLM-assisted evaluation (LLM-as-a-Judge)** coupled with curated golden datasets in **LangSmith** enables continuous quality tracking, regression testing, and objective model comparisons.

---

![Chatbot Evaluation Workflow](workflow_chatbot_evaluation.png)

<details>
<summary><b>🔍 Click to Expand Interactive Mermaid Architecture Diagram</b></summary>

```mermaid
flowchart TD
    classDef dataset fill:#EBF5FB,stroke:#2980B9,stroke-width:2px,color:#1B4F72;
    classDef bot fill:#F4ECF7,stroke:#8E44AD,stroke-width:2px,color:#512E5F;
    classDef eval fill:#FEF9E7,stroke:#F39C12,stroke-width:2px,color:#7D6608;
    classDef platform fill:#D5F5E3,stroke:#27AE60,stroke-width:2px,color:#145A32;
    classDef table fill:#FDEDEC,stroke:#E74C3C,stroke-width:2px,color:#78281F;

    DS["📋 Curated Test Dataset<br/>(Golden Questions & Reference Answers)"]:::dataset
    
    DS --> BotA["⚡ Candidate A: Concise Chatbot<br/>(Direct, single-sentence response)"]:::bot
    DS --> BotB["📖 Candidate B: Detailed Chatbot<br/>(Educational, step-by-step response)"]:::bot
    
    BotA --> Suite["⚖️ LangSmith Evaluation Suite"]:::eval
    BotB --> Suite
    
    subgraph Suite ["⚖️ Multi-Metric Evaluation Panel"]
        J1["👨‍🏫 LLM-as-a-Judge: Correctness<br/>(Structured step-by-step reasoning)"]:::eval
        J2["📏 Rule-Based: Concision<br/>(Length ratio vs ground truth)"]:::eval
        J3["🤝 LLM-as-a-Judge: Helpfulness & Tone<br/>(Polite, clear, actionable tone)"]:::eval
    end
    
    Suite --> LS["🌐 LangSmith Observability Platform<br/>(Traces, Session Diffs & Run Analytics)"]:::platform
    Suite --> Scorecard["📊 Executive Scorecard & Comparison Table"]:::table
```

</details>

---

### 📚 What You'll Master
1. **The Chatbot Evaluation Lifecycle**: Why manual spot-checks fail and how automated evaluation pipelines protect production quality.
2. **Golden Dataset Management**: Creating and versioning test cases with realistic questions and expected answers in LangSmith.
3. **Candidate Prompt & Model Exploration**: Deploying contrasting assistant personas (Fast Concise vs Deep Educational).
4. **LLM-as-a-Judge Architecture**: Crafting structured evaluators where the model explains its reasoning before assigning a grade.
5. **Deterministic Heuristics**: Implementing rule-based concision checks that execute instantaneously at zero inference cost.
6. **Automated LangSmith Benchmarks**: Running `client.evaluate` to grade entire test suites asynchronously.
7. **Side-by-Side Experiment Scorecards**: Quantitatively comparing response times, accuracy pass rates, and concision ratios.


## ⚙️ Section 1: Environment & API Configuration

Before running our evaluation experiments, we load our environment variables and verify that both the **Groq API** (which powers our high-speed candidate chatbots and evaluator judges) and **LangSmith** (which stores our datasets and tracks evaluation runs) are configured properly.


In [1]:
"""
In this setup step, we prepare our Python environment and bring in the necessary helper tools.
We read our secret API credentials from the local .env file so our program can securely communicate
with Groq to generate answers and with LangSmith to log our benchmark scores.
We also make sure our terminal cleanly displays unicode checkmarks and hides unnecessary warning messages.
"""

import os
import sys
import time
import json
import warnings
import logging
import pandas as pd
from dotenv import load_dotenv

# Reconfigure standard output for UTF-8 on Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)

load_dotenv()

# Verify required credentials
groq_key = os.getenv("GROQ_API_KEY")
langsmith_key = os.getenv("LANGSMITH_API_KEY")

print("=== 🔐 System Verification ===")
print(f"Groq API Key Active:       {'✅ Connected' if groq_key else '❌ Missing'}")
print(f"LangSmith API Key Active:  {'✅ Connected' if langsmith_key else '❌ Missing'}")


=== 🔐 System Verification ===
Groq API Key Active:       ✅ Connected
LangSmith API Key Active:  ✅ Connected


## 📋 Section 2: Creating the Golden Evaluation Dataset

A **Golden Dataset** is the single source of truth for evaluation. It consists of representative user questions paired with validated, human-vetted reference answers. Whenever prompt templates, system instructions, or underlying LLM models change, we re-run this exact dataset to detect quality regressions immediately.


In [2]:
"""
Think of this step like writing an exam paper along with an official answer key.
We connect to our LangSmith workspace and create a dedicated test dataset named 'Chatbot-Evaluation-Suite'.
We then upload five realistic technical questions along with their ideal reference answers.
Every candidate chatbot we test will face these exact same questions under identical conditions.
"""

from langsmith import Client

client = Client()
dataset_name = "Chatbot-Evaluation-Suite"

# Clean up older runs if present to keep the workspace tidy
try:
    for existing_ds in client.list_datasets(dataset_name=dataset_name):
        client.delete_dataset(dataset_id=existing_ds.id)
except Exception:
    pass

# Create the fresh evaluation dataset in LangSmith
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Standardized evaluation benchmark for testing LLM chatbot accuracy, concision, and tone."
)

# Define our golden Q&A test cases
test_examples = [
    {
        "inputs": {"question": "What is LangChain and what is its primary purpose?"},
        "outputs": {"answer": "LangChain is an open-source development framework designed to build applications powered by large language models by connecting models to data sources and tools."}
    },
    {
        "inputs": {"question": "How does a Vector Database differ from a traditional relational database?"},
        "outputs": {"answer": "A vector database indexes and retrieves high-dimensional numerical embeddings using semantic similarity, whereas relational databases query structured tabular data using exact keyword matches."}
    },
    {
        "inputs": {"question": "What is Retrieval-Augmented Generation (RAG)?"},
        "outputs": {"answer": "RAG is a technique that enhances LLM responses by retrieving relevant external knowledge from an external database and providing it as context to the model."}
    },
    {
        "inputs": {"question": "What causes hallucinations in large language models?"},
        "outputs": {"answer": "Hallucinations occur when an LLM produces plausible-sounding but factually incorrect information due to gaps, biases, or statistical patterns in its training data."}
    },
    {
        "inputs": {"question": "What is the purpose of LangSmith in production AI applications?"},
        "outputs": {"answer": "LangSmith is a platform for debugging, monitoring, tracing, evaluating, and testing LLM applications across development and production environments."}
    }
]

# Upload the test examples to the LangSmith dataset
client.create_examples(
    dataset_id=dataset.id,
    examples=test_examples
)

print(f"✅ Successfully created dataset: '{dataset_name}'")
print(f"📋 Dataset ID: {dataset.id}")
print(f"📊 Total Golden Examples Uploaded: {len(test_examples)}")


✅ Successfully created dataset: 'Chatbot-Evaluation-Suite'
📋 Dataset ID: 700cc9a2-630f-4648-9bf5-bdd6e3758922
📊 Total Golden Examples Uploaded: 5


## 🤖 Section 3: Defining Candidate Chatbots

To illustrate how evaluation guides engineering decisions, we build two distinct candidate chatbots using **`qwen/qwen3.8-27b`** on Groq:

1. **Candidate A (Concise Fast Bot)**: System prompt instructs the model to provide extremely short, single-sentence answers.
2. **Candidate B (Detailed Educational Bot)**: System prompt instructs the model to explain concepts thoroughly with practical context and examples.


In [3]:
"""
Here, we build two different chatbot personalities that we want to put to the test.
Both bots use the same fast underlying model, but their instructions are very different:
- Candidate A is built for people who want quick, one-sentence punchy answers.
- Candidate B is built for learners who want thorough explanations with rich context.
We also run a test question on both to see the immediate difference in their answering styles.
"""

from langchain_groq import ChatGroq

# Initialize high-speed model client
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    groq_api_key=groq_key,
    temperature=0.1
)

# 1. Candidate A: Concise Assistant
def candidate_a_concise(question: str) -> str:
    prompt = f"""You are a concise, direct AI assistant.
Answer the following question in exactly ONE short, punchy sentence. Do not elaborate.

Question: {question}
Answer:"""
    response = llm.invoke(prompt)
    return response.content.strip()

# 2. Candidate B: Detailed Educational Assistant
def candidate_b_detailed(question: str) -> str:
    prompt = f"""You are a patient, expert teacher.
Answer the following question thoroughly in 2 to 3 informative sentences. Explain the core mechanism clearly.

Question: {question}
Answer:"""
    response = llm.invoke(prompt)
    return response.content.strip()

# Run a quick demonstration on a sample question
sample_q = "What is an embedding vector?"
print("=== 🧪 Prototype Demonstration ===")
print("Query:", sample_q)
print("-" * 55)
print("Candidate A (Concise):\n", candidate_a_concise(sample_q))
print("-" * 55)
print("Candidate B (Detailed):\n", candidate_b_detailed(sample_q))


=== 🧪 Prototype Demonstration ===
Query: What is an embedding vector?
-------------------------------------------------------


Candidate A (Concise):
 An embedding vector is a numerical array that encodes the semantic meaning of data into a continuous vector space.
-------------------------------------------------------


Candidate B (Detailed):
 An embedding vector is a dense, fixed-length numerical array that represents a discrete item, such as a word or image, in a continuous multi-dimensional space. The core mechanism relies on the geometric property that semantically similar items are mapped to nearby coordinates, allowing complex relationships and meanings to be captured through simple mathematical operations like distance calculation.


## ⚖️ Section 4: Engineering the Evaluation Suite (LLM-as-a-Judge & Heuristics)

A robust evaluation harness combines multiple grading dimensions:

1. **Factual Correctness (LLM-as-a-Judge)**: A teacher model reads the question, reference answer, and predicted answer. To prevent erratic scoring, we enforce **structured output** where the model must generate an `explanation` (chain of thought) *before* returning the boolean score `is_correct`.
2. **Concision (Rule-Based Heuristic)**: Measures whether the bot stayed within a reasonable character length budget relative to the reference answer.
3. **Helpfulness & Clarity (LLM-as-a-Judge)**: Grades whether the response is polite, easily understood, and free of unnecessary fluff.


In [4]:
"""
In this cell, we build our primary judge: an AI professor that grades factual correctness.
Notice how we define a structured answer format: the judge is required to write out its
reasoning step-by-step FIRST, and only then award a pass or fail grade.
This mirrors how human teachers grade exam papers, dramatically reducing accidental mistakes.
"""

from typing_extensions import TypedDict, Annotated

# Define the structured output schema for the evaluator
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Step-by-step reasoning comparing student answer against the ground truth reference."]
    is_correct: Annotated[bool, ..., "True if the student answer is factually accurate relative to the reference answer, False otherwise."]

# Attach the structured schema to our grader model
correctness_grader = llm.with_structured_output(CorrectnessGrade)

def eval_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    prompt = f"""You are an expert professor grading a student's technical exam response.

QUESTION:
{inputs['question']}

GROUND TRUTH REFERENCE ANSWER:
{reference_outputs['answer']}

STUDENT'S SUBMITTED ANSWER:
{outputs['answer']}

GRADING CRITERIA:
1. Grade the student answer ONLY on factual consistency relative to the ground truth.
2. It is acceptable if the student uses different phrasing or adds harmless context, as long as it does not contradict the ground truth.
3. If the student answer states false information or fails to address the question, mark it incorrect.
"""
    grade = correctness_grader.invoke(prompt)
    return {
        "score": 1 if grade["is_correct"] else 0,
        "comment": grade["explanation"]
    }

print("✅ Factual Correctness Evaluator initialized with structured reasoning.")


✅ Factual Correctness Evaluator initialized with structured reasoning.


In [5]:
"""
Here, we introduce two complementary evaluators to our grading panel:
1. A fast, rule-based length checker: It verifies that the answer is reasonably brief and not overly verbose.
   Because it uses simple arithmetic on string lengths, it runs instantly at zero extra dollar cost!
2. A tone and helpfulness judge: An AI evaluator that checks whether the bot speaks in a friendly,
   polite, and approachable manner that makes the user feel supported.
"""

class HelpfulnessGrade(TypedDict):
    explanation: Annotated[str, ..., "Detailed reasoning on tone, clarity, and helpfulness."]
    is_helpful: Annotated[bool, ..., "True if tone is polite, professional, and directly helpful."]

helpfulness_grader = llm.with_structured_output(HelpfulnessGrade)

# 1. Rule-Based Concision Evaluator
def eval_concision(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    student_len = len(outputs["answer"].strip())
    ref_len = len(reference_outputs["answer"].strip())
    
    # Consider concise if student answer is under 1.8x the reference answer length
    ratio = round(student_len / max(ref_len, 1), 2)
    is_concise = ratio <= 1.8
    
    return {
        "score": 1 if is_concise else 0,
        "comment": f"Length ratio: {ratio}x (Student: {student_len} chars vs Reference: {ref_len} chars)"
    }

# 2. LLM Tone & Helpfulness Evaluator
def eval_helpfulness(inputs: dict, outputs: dict) -> dict:
    prompt = f"""You are evaluating the customer service quality of an AI chatbot response.

User Question: {inputs['question']}
Bot Response: {outputs['answer']}

Criteria:
- Is the tone professional, approachable, and respectful?
- Is the response free from confusing jargon or robotic condescension?
"""
    grade = helpfulness_grader.invoke(prompt)
    return {
        "score": 1 if grade["is_helpful"] else 0,
        "comment": grade["explanation"]
    }

print("✅ Rule-Based Concision and LLM Helpfulness Evaluators initialized.")


✅ Rule-Based Concision and LLM Helpfulness Evaluators initialized.


## 🚀 Section 5: Running the Automated Benchmark in LangSmith

Now we invoke `client.evaluate()`. Under the hood, LangSmith:
1. Pulls each test case from the dataset.
2. Sends the question to our candidate function (`target_wrapper`).
3. Passes the inputs, chatbot output, and reference answer to all evaluators in parallel.
4. Records execution latencies, token consumption, and feedback metrics.
5. Returns a rich Python dataset that converts directly into a Pandas DataFrame.


In [6]:
"""
Now we conduct the official exam for Candidate A (our Concise Bot)!
LangSmith feeds each of the five questions to Candidate A, collects its responses,
hands the answers to our three evaluators, and logs all traces to our LangSmith project.
Finally, we display the scorecard as a neat table to inspect every grade.
"""

# Wrapper function matching LangSmith's input contract
def run_concise_bot(inputs: dict) -> dict:
    ans = candidate_a_concise(inputs["question"])
    return {"answer": ans}

print("🚀 Starting Evaluation for Candidate A (Concise Bot)...")
t0 = time.time()

eval_results_a = client.evaluate(
    run_concise_bot,
    data=dataset_name,
    evaluators=[eval_correctness, eval_concision, eval_helpfulness],
    experiment_prefix="concise-bot-benchmark",
    metadata={"version": "1.0", "persona": "concise"}
)

elapsed_a = time.time() - t0
df_a = eval_results_a.to_pandas()
print(f"✅ Finished in {elapsed_a:.2f} seconds!")

# Display key columns
cols = ["inputs.question", "outputs.answer", "feedback.eval_correctness", "feedback.eval_concision", "feedback.eval_helpfulness"]
df_a_display = df_a[[c for c in cols if c in df_a.columns]]
df_a_display


🚀 Starting Evaluation for Candidate A (Concise Bot)...


View the evaluation results for experiment: 'concise-bot-benchmark-7e07d7e2' at:
https://smith.langchain.com/o/24e3cc69-95f0-469c-bf68-ac531fe7f46f/datasets/700cc9a2-630f-4648-9bf5-bdd6e3758922/compare?selectedSessions=78ee2e0c-0460-49a6-ad93-00c37e72b30c




0it [00:00, ?it/s]

1it [00:01,  1.92s/it]

2it [00:04,  2.20s/it]

...

4it [00:40, 12.65s/it]

5it [00:52, 12.59s/it]

5it [00:53, 10.67s/it]

✅ Finished in 55.39 seconds!


,inputs.question,outputs.answer,feedback.eval_correctness,feedback.eval_concision,feedback.eval_helpfulness
0,How does a Vector Database differ from a tradi...,Vector databases store and query high-dimensio...,1,1,1
1,What causes hallucinations in large language m...,They are the inevitable byproduct of probabili...,1,1,1
2,What is Retrieval-Augmented Generation (RAG)?,RAG is an AI technique that grounds large lang...,1,1,1
3,What is LangChain and what is its primary purp...,LangChain is a development framework that simp...,1,1,1
4,What is the purpose of LangSmith in production...,"LangSmith provides observability, evaluation, ...",1,1,1


In [7]:
"""
Next, we run the identical exam on Candidate B (our Detailed Bot)!
Because the questions and the grading criteria remain exactly the same,
this allows us to objectively observe how being more detailed affects accuracy and length.
"""

def run_detailed_bot(inputs: dict) -> dict:
    ans = candidate_b_detailed(inputs["question"])
    return {"answer": ans}

print("🚀 Starting Evaluation for Candidate B (Detailed Bot)...")
t0 = time.time()

eval_results_b = client.evaluate(
    run_detailed_bot,
    data=dataset_name,
    evaluators=[eval_correctness, eval_concision, eval_helpfulness],
    experiment_prefix="detailed-bot-benchmark",
    metadata={"version": "1.0", "persona": "detailed"}
)

elapsed_b = time.time() - t0
df_b = eval_results_b.to_pandas()
print(f"✅ Finished in {elapsed_b:.2f} seconds!")

# Display key columns
df_b_display = df_b[[c for c in cols if c in df_b.columns]]
df_b_display


🚀 Starting Evaluation for Candidate B (Detailed Bot)...


View the evaluation results for experiment: 'detailed-bot-benchmark-a27bf1f5' at:
https://smith.langchain.com/o/24e3cc69-95f0-469c-bf68-ac531fe7f46f/datasets/700cc9a2-630f-4648-9bf5-bdd6e3758922/compare?selectedSessions=56395797-7f83-45db-9af6-53cec62c7c9b




0it [00:00, ?it/s]

1it [00:26, 26.52s/it]

2it [00:56, 28.56s/it]

...

4it [01:42, 25.27s/it]

5it [02:06, 24.89s/it]

5it [02:07, 25.46s/it]

✅ Finished in 129.02 seconds!


,inputs.question,outputs.answer,feedback.eval_correctness,feedback.eval_concision,feedback.eval_helpfulness
0,How does a Vector Database differ from a tradi...,A traditional relational database stores data ...,1,0,1
1,What causes hallucinations in large language m...,Hallucinations in large language models primar...,1,0,1
2,What is Retrieval-Augmented Generation (RAG)?,Retrieval-Augmented Generation (RAG) is a hybr...,1,0,1
3,What is LangChain and what is its primary purp...,LangChain is a modular development framework d...,1,0,1
4,What is the purpose of LangSmith in production...,LangSmith serves as a comprehensive observabil...,1,0,1


## 📊 Section 6: Side-by-Side Comparative Scorecard

In production, engineering decisions require comparing key trade-offs:
- **Accuracy Pass Rate**: Did the concise prompt sacrifice factual completeness?
- **Concision Compliance**: Did the detailed prompt ramble excessively?
- **Helpfulness**: Which bot scored higher on user satisfaction?
- **Speed & Latency**: How much faster was the concise model?


In [8]:
"""
Here, we crunch the numbers from both runs and print out an executive comparison summary.
We calculate the overall passing rate for factual correctness, the concision compliance rate,
the helpfulness score, and the average time it took each bot to answer a question.
This gives engineering teams the exact evidence they need to choose the best bot for their users.
"""

# Helper to safely extract metric averages
def get_avg(df, col):
    if col in df.columns:
        return df[col].astype(float).mean() * 100
    return 0.0

summary_data = {
    "Evaluation Metric": [
        "Factual Correctness Pass Rate",
        "Concision Compliance Rate",
        "Helpfulness / Tone Score",
        "Average Answer Length (chars)"
    ],
    "Candidate A (Concise Bot)": [
        f"{get_avg(df_a, 'feedback.eval_correctness'):.1f}%",
        f"{get_avg(df_a, 'feedback.eval_concision'):.1f}%",
        f"{get_avg(df_a, 'feedback.eval_helpfulness'):.1f}%",
        f"{df_a['outputs.answer'].str.len().mean():.0f} chars"
    ],
    "Candidate B (Detailed Bot)": [
        f"{get_avg(df_b, 'feedback.eval_correctness'):.1f}%",
        f"{get_avg(df_b, 'feedback.eval_concision'):.1f}%",
        f"{get_avg(df_b, 'feedback.eval_helpfulness'):.1f}%",
        f"{df_b['outputs.answer'].str.len().mean():.0f} chars"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("==================================================================")
print("🏆 EXECUTIVE CHATBOT BENCHMARK SCORECARD")
print("==================================================================")
print(summary_df.to_string(index=False))
print("==================================================================")


🏆 EXECUTIVE CHATBOT BENCHMARK SCORECARD
            Evaluation Metric Candidate A (Concise Bot) Candidate B (Detailed Bot)
Factual Correctness Pass Rate                    100.0%                     100.0%
    Concision Compliance Rate                    100.0%                       0.0%
     Helpfulness / Tone Score                    100.0%                     100.0%
Average Answer Length (chars)                 161 chars                  589 chars


## 🎯 Section 7: Key Takeaways & Production Best Practices

| Best Practice | Why It Matters in Production | Implementation Pattern |
|:---|:---|:---|
| **Separate Datasets from Code** | Allows non-technical domain experts to curate test cases without touching code. | LangSmith Datasets with versioning |
| **Reasoning-First Evaluators** | Forcing models to write an explanation before assigning scores prevents hasty errors. | `with_structured_output` with `TypedDict` schema |
| **Combine Rule-Based with LLMs** | Rule-based checks (length, regex, profanity) cost $0 and catch simple errors instantly. | Deterministic Python functions alongside LLM judges |
| **Continuous Regression CI/CD** | Ensures new prompt revisions or model switches never degrade core answering ability. | Trigger `client.evaluate` on GitHub Actions or releases |
